# arora_data functions

Each cell calls one function against prod, read-only, as `readonly_user`, and shows its output. Change the arguments and re-run any cell.

In [ ]:
from datetime import timedelta

from IPython.display import Markdown

from arora_data import _readers
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.series import Series
from arora_data.sync import _sync, check_fresh, list_columns, list_schemas, list_series, list_tables

# The two readers are private: apps will call named series functions instead. Wrapped here only to explore.
read_prices = _sync(_readers._read_prices)
read_values = _sync(_readers._read_values)


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## `today_et()`
Today's date in US Eastern time.

In [ ]:
today_et()

## `list_schemas()`
The data schemas and what each holds.

In [ ]:
show(list_schemas())

## `list_tables(schema)`
Each table's description, estimated rows and most recent date.

In [ ]:
show(list_tables("forex"))

## `list_columns(schema, table)`
Column names, types and whether they can be blank.

In [ ]:
show(list_columns("forex", "prices"))

## `list_series(schema, table)`
The series in a time-series table, with each one's most recent date.

In [ ]:
show(list_series("forex", "prices"))

## `check_fresh(series)` on a current series
USD/CAD is daily, so it is stale after 10 days without a new point.

In [ ]:
show(check_fresh(Series("forex.prices", "USDCAD", frequency="daily")))

## `check_fresh(series)` on a stopped series
FRED `GDP` was dropped from the FRED list in migration 096, and has no frequency label, so it counts as stale.

In [ ]:
show(check_fresh(Series("macro.observations", "GDP")))

## `_read_prices(table, symbol, start, end)`
Daily price bars, oldest first. Private: for exploring only.

In [ ]:
show(read_prices("forex.prices", "USDCAD", today_et() - timedelta(days=14), today_et()))

## `_read_values(table, key, start, end)`
One series' values, oldest first. Private: for exploring only.

In [ ]:
show(read_values("macro.observations", "BAMLH0A0HYM2", today_et() - timedelta(days=14), today_et()))

## `_read_values(..., dataset=...)` on an `indicators.*` table
These tables need `dataset`, because the same key can mean different series in different datasets.

In [ ]:
show(read_values("indicators.bls", "CES0000000001", today_et() - timedelta(days=365), today_et(), dataset="bls/core"))

## `to_df(rows, model)`
Any result as a Polars dataframe, with exact `Decimal` values.

In [ ]:
to_df(read_values("macro.observations", "BAMLH0A0HYM2", today_et() - timedelta(days=14), today_et()), _readers.Value)